# Entrenamiento YOLO26 - Laboratorio Rumiologia

Fine-tuning de `yolo26n.pt` para detectar los equipos del laboratorio,
y exportacion a TFLite para la app Android.

**Antes de empezar:** `Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU`.

Ejecuta las celdas en orden. La celda 4 (entrenamiento) es la larga.


## 1. Verificar que hay GPU

Si no aparece una Tesla T4, cambia el tipo de entorno antes de seguir:
entrenar en CPU tarda entre 10 y 30 veces mas.


In [ ]:
!nvidia-smi


## 2. Montar Drive y traer el dataset al disco local

El dataset se copia como .zip y se descomprime en `/content`.
Leer miles de imagenes sueltas montadas desde Drive puede tardar mas que el
propio entrenamiento, porque cada archivo es una peticion de red.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Laboratorio_Rumiologia'

!cp '{BASE}/dataset.zip' /content/
!unzip -q -o /content/dataset.zip -d /content/

# Comprobacion de que la estructura llego completa
!echo '--- train:' && ls /content/dataset/images/train | wc -l
!echo '--- val:'   && ls /content/dataset/images/val   | wc -l
!echo '--- test:'  && ls /content/dataset/images/test  | wc -l
!cat /content/dataset/data.yaml


## 3. Instalar Ultralytics


In [ ]:
!pip install -q -U ultralytics

import ultralytics
ultralytics.checks()


## 4. Entrenar

`YOLO('yolo26n.pt')` descarga los pesos preentrenados en COCO; `.train()` sobre
ellos es fine-tuning para tus clases. No hace falta nada mas para partir de lo
ya aprendido.

Ajustes segun tu caso:
- `batch=8` si sale error de memoria (OOM).
- `freeze=10` si tienes menos de ~500 imagenes: congela el backbone, converge
  con menos datos y sobreajusta menos.
- `imgsz` debe ser el MISMO valor al exportar y en Android.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

resultados = model.train(
    data='/content/dataset/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    patience=30,          # early stopping: corta si no mejora en 30 epocas
    device=0,
    workers=8,
    cache='ram',          # quitalo si el dataset no cabe en RAM
    seed=0,
    project=f'{BASE}/runs',   # checkpoints a Drive: sobreviven desconexiones
    name='rumio_v1',
)


## 4b. Reanudar si Colab se desconecto

Solo si la celda anterior se corto a medias. Retoma desde el ultimo checkpoint
sin perder lo entrenado.


In [ ]:
# from ultralytics import YOLO
# YOLO(f'{BASE}/runs/rumio_v1/weights/last.pt').train(resume=True)


## 5. Validar sobre el split de test

Mira el mAP50 **por clase**, no solo el global. Si una clase queda muy por debajo
del resto, la solucion son mas fotos de esa clase, no mas epocas.


In [ ]:
from ultralytics import YOLO

best = YOLO(f'{BASE}/runs/rumio_v1/weights/best.pt')
metricas = best.val(data='/content/dataset/data.yaml', split='test')


### Curvas y matriz de confusion

La matriz de confusion dice *que* clases se confunden entre si: es la pista mas
util para saber que fotos hacen falta.


In [ ]:
from IPython.display import Image, display
import glob

for p in ['confusion_matrix.png', 'results.png', 'val_batch0_pred.jpg']:
    for f in glob.glob(f'{BASE}/runs/rumio_v1/**/{p}', recursive=True):
        print(f)
        display(Image(filename=f, width=760))


## 6. Probar sobre imagenes sueltas

Comprobacion visual antes de exportar.


In [ ]:
import glob
from IPython.display import Image, display

muestra = sorted(glob.glob('/content/dataset/images/test/*'))[:3]
best.predict(muestra, conf=0.5, save=True, project='/content/pred', name='demo')

for f in sorted(glob.glob('/content/pred/demo/*')):
    display(Image(filename=f, width=640))


## 7. Exportar a TFLite para Android

**Dos trampas comprobadas en Ultralytics 8.4.131, ambas necesarias:**

1. **Carga una instancia NUEVA del modelo.** Si exportas el mismo objeto que ya
   paso por `.val()` o `.predict()`, falla con `KeyError: 'feats'` — el estado
   interno de la cabeza queda alterado y el decodificador no lo soporta.
2. **No uses `int8=True`.** La cuantizacion desactiva la rama end-to-end de
   YOLO26 y rompe la exportacion. Sin cuantizar se conserva la salida sin NMS,
   que es justo la ventaja del modelo.

`format='litert'` sustituye a `format='tflite'`, obsoleto desde la 8.4.83.
Genera el mismo archivo .tflite (~9 MB en float32).


In [ ]:
from ultralytics import YOLO

# Instancia limpia: no reutilices 'best' de las celdas anteriores
modelo_exportar = YOLO(f'{BASE}/runs/rumio_v1/weights/best.pt')
rutas = modelo_exportar.export(format='litert', imgsz=640)
print(rutas)

!ls -lh {BASE}/runs/rumio_v1/weights/


## 8. Inspeccionar la firma del modelo exportado

**No te saltes esta celda.** Define como se escribe el post-procesado en Android.

En este proyecto dio:

```
ENTRADAS: [1, 3, 640, 640]  float32   <- NCHW (canales primero, orden de PyTorch)
SALIDAS : [1, 300, 6]       float32   <- end-to-end: x1,y1,x2,y2,score,clase
```

La entrada NCHW obliga a llenar el buffer por planos completos (todos los R,
luego los G, luego los B) en vez de pixel a pixel. La salida end-to-end evita
tener que implementar NMS en Java. `Detector.java` detecta ambas cosas solo.


In [ ]:
import tensorflow as tf, glob

rutas = sorted(glob.glob(f'{BASE}/runs/rumio_v1/weights/**/*.tflite', recursive=True))
print('encontrados:', rutas)

interp = tf.lite.Interpreter(model_path=rutas[0])
interp.allocate_tensors()

print('\n--- ENTRADAS ---')
for d in interp.get_input_details():
    print(d['name'], d['shape'], d['dtype'], d['quantization'])

print('\n--- SALIDAS ---')
for d in interp.get_output_details():
    print(d['name'], d['shape'], d['dtype'], d['quantization'])


## 9. Descargar

Dos archivos con destinos distintos:

| Archivo | Destino |
|---|---|
| `best.tflite` | `app/src/main/assets/` **renombrado a `model.tflite`** |
| `best.pt` | `ml/models/` — respaldo para reentrenar |


In [ ]:
from google.colab import files
import glob

rutas = sorted(glob.glob(f'{BASE}/runs/rumio_v1/weights/**/*.tflite', recursive=True))
print('descargando:', rutas[0])
files.download(rutas[0])

files.download(f'{BASE}/runs/rumio_v1/weights/best.pt')


---

## 10. Reentrenar mas adelante con fotos nuevas

Cuando agregues fotos (sobre todo de `ankom_estufa`, que quedo con 7 imagenes
de entrenamiento), el proceso es el mismo hasta aqui: etiquetar en Label Studio,
`split_dataset.py`, `check_dataset.py`, comprimir y subir el `dataset.zip` nuevo.

**`data.yaml` no cambia** mientras sean las mismas 7 clases en el mismo orden.

Al entrenar hay dos opciones:

- **Partir de `best.pt`** (la celda de abajo): aprovecha lo ya aprendido y
  converge mas rapido. Util cuando solo agregas fotos de las clases que ya
  existen.
- **Partir de `yolo26n.pt` otra vez**: mas lento pero sin arrastrar sesgos del
  entrenamiento anterior. Preferible si cambiaste mucho el dataset o corregiste
  etiquetas mal puestas.

Usa un `name` distinto (`rumio_v2`) para no pisar los resultados anteriores y
poder comparar.


In [ ]:
# Reentrenamiento partiendo del modelo ya entrenado
from ultralytics import YOLO

modelo = YOLO(f'{BASE}/runs/rumio_v1/weights/best.pt')

modelo.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=30,
    device=0,
    workers=8,
    cache='ram',
    seed=0,
    project=f'{BASE}/runs',
    name='rumio_v2',        # nombre nuevo: no pisa el entrenamiento anterior
)

# Despues repite las celdas 5 a 9 cambiando rumio_v1 por rumio_v2
